# 📈 Curso de Big Data e IA — Módulo 1
## Bloque 4: Visualización de datos y Business Intelligence

**Duración estimada:** 3 horas · **Caso práctico:** Real Sporting de Gijón, La Liga 2015/16

**Datos:** [StatsBomb Open Data](https://github.com/statsbomb/open-data). Si publicas o compartes análisis basados en estos datos, indica StatsBomb como fuente y usa su logo (disponible en su *media pack*).

**Objetivos**
- Entender el papel de la visualización en la toma de decisiones (BI).
- Construir un mini **cuadro de mando** con Plotly a partir de los datos limpios del Notebook 1.
- Conocer el panorama de herramientas profesionales de BI (Power BI, Tableau, Qlik...).

**Antes de empezar:** sube a Colab (panel de archivos de la izquierda) los dos ficheros generados en el Notebook 1:
`sporting_partidos.csv` y `sporting_eventos.csv`.

## 1. De datos a decisiones

Un **cuadro de mando** convierte datos en **KPIs** que una persona no técnica entiende de un vistazo.
En fútbol, un cuerpo técnico se haría preguntas como:

- ¿Cuántos puntos sumamos y cómo evolucionó la temporada?
- ¿Somos mejores en casa que fuera?
- ¿Creamos más ocasiones de las que concedemos? (xG a favor frente a xG en contra)
- ¿Quién remata más y quién aprovecha mejor sus ocasiones?

Herramientas profesionales habituales: Power BI, Tableau, Qlik, Looker... Hoy hacemos lo mismo con código.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

EQUIPO = "Sporting Gijón"

try:
    partidos = pd.read_csv("sporting_partidos.csv", parse_dates=["fecha"])
    eventos = pd.read_csv("sporting_eventos.csv")
except FileNotFoundError:
    raise SystemExit("Sube sporting_partidos.csv y sporting_eventos.csv (generados en el Notebook 1) al panel de archivos de Colab.")

partidos["puntos_acum"] = partidos["puntos"].cumsum()
tiros = eventos[eventos["type"] == "Shot"].copy()
tiros["gol"] = (tiros["shot_outcome"] == "Goal").astype(int)

print(f"{len(partidos)} partidos · {len(eventos):,} eventos · {len(tiros)} tiros")

## 2. KPIs generales

In [ ]:
puntos = partidos["puntos"].sum()
victorias = (partidos["resultado"] == "V").sum()
gf, gc = partidos["goles_favor"].sum(), partidos["goles_contra"].sum()

print(f"Puntos: {puntos}")
print(f"Victorias: {victorias} de {len(partidos)} ({victorias / len(partidos) * 100:.0f}%)")
print(f"Goles a favor: {gf} | Goles en contra: {gc} | Diferencia: {gf - gc}")

## 3. KPI 1 — Evolución de la temporada

In [ ]:
fig = px.line(partidos, x="jornada", y="puntos_acum", markers=True,
              hover_data=["rival", "resultado"],
              title=f"Puntos acumulados del {EQUIPO} (La Liga 2015/16)")
fig.update_layout(xaxis_title="Jornada", yaxis_title="Puntos")
fig.show()

## 4. KPI 2 — Local frente a visitante

In [ ]:
por_campo = (partidos.groupby("local")[["puntos", "goles_favor", "goles_contra"]].sum()
             .rename(index={True: "Local", False: "Visitante"})
             .reset_index().rename(columns={"local": "campo"}))

fig = px.bar(por_campo.melt(id_vars="campo", var_name="métrica", value_name="valor"),
             x="métrica", y="valor", color="campo", barmode="group", text_auto=True,
             title="Puntos y goles: local frente a visitante")
fig.show()

## 5. KPI 3 — ¿Creamos más de lo que concedemos? (xG)

**xG** (*expected goals*) mide la calidad de las ocasiones. Si en un partido el xG a favor supera al xG en contra,
el equipo mereció más. Cada punto es un partido: por encima de la diagonal, el Sporting creó más peligro que el rival.

In [ ]:
xg = tiros.groupby(["match_id", "team"])["shot_xg"].sum().reset_index()
xg_favor = xg[xg["team"] == EQUIPO].rename(columns={"shot_xg": "xg_favor"})[["match_id", "xg_favor"]]
xg_contra = xg[xg["team"] != EQUIPO].rename(columns={"shot_xg": "xg_contra"})[["match_id", "xg_contra"]]

xg_partido = (partidos[["match_id", "jornada", "rival", "resultado"]]
              .merge(xg_favor, on="match_id", how="left")
              .merge(xg_contra, on="match_id", how="left")
              .fillna({"xg_favor": 0, "xg_contra": 0}))

fig = px.scatter(xg_partido, x="xg_contra", y="xg_favor", color="resultado",
                 hover_name="rival", hover_data=["jornada"],
                 color_discrete_map={"V": "#2E8B57", "E": "#E0A800", "D": "#D1495B"},
                 title="xG a favor frente a xG en contra, por partido")
limite = float(max(xg_partido["xg_favor"].max(), xg_partido["xg_contra"].max())) + 0.3
fig.add_shape(type="line", x0=0, y0=0, x1=limite, y1=limite, line=dict(dash="dash", color="gray"))
fig.update_layout(xaxis_title="xG en contra", yaxis_title="xG a favor")
fig.show()

print(f"xG a favor: {xg_partido['xg_favor'].sum():.1f}  |  xG en contra: {xg_partido['xg_contra'].sum():.1f}")

## 6. KPI 4 — Rematadores: goles reales frente a esperados

In [ ]:
tiros_sp = tiros[tiros["team"] == EQUIPO]
jugadores = (tiros_sp.groupby("player")
             .agg(tiros=("shot_xg", "size"), goles=("gol", "sum"), xg=("shot_xg", "sum"))
             .sort_values("tiros", ascending=False).head(8).reset_index())
jugadores["xg"] = jugadores["xg"].round(1)

fig = px.bar(jugadores.melt(id_vars=["player", "tiros"], value_vars=["goles", "xg"], var_name="medida", value_name="valor"),
             x="player", y="valor", color="medida", barmode="group",
             title="Goles reales frente a goles esperados (xG) — top 8 por número de tiros")
fig.update_layout(xaxis_title="", yaxis_title="Goles")
fig.show()
jugadores

## 7. Mapa de tiros interactivo

In [ ]:
tiros_sp = tiros_sp.copy()
tiros_sp["resultado_tiro"] = np.where(tiros_sp["gol"] == 1, "Gol", "Sin gol")

fig = px.scatter(tiros_sp, x="x", y="y", color="resultado_tiro", size="shot_xg",
                 hover_data=["player", "minute", "shot_body_part"],
                 color_discrete_map={"Gol": "#D1495B", "Sin gol": "#2E86AB"},
                 title="Tiros del Sporting (tamaño = xG)")
# Líneas del campo (120 x 80). El eje y se invierte para coincidir con los datos
for x0, y0, x1, y1 in [(0, 0, 120, 80), (102, 18, 120, 62), (114, 30, 120, 50)]:
    fig.add_shape(type="rect", x0=x0, y0=y0, x1=x1, y1=y1, line=dict(color="gray"))
fig.add_shape(type="line", x0=60, y0=0, x1=60, y1=80, line=dict(color="gray"))
fig.update_xaxes(range=[55, 122], title="")
fig.update_yaxes(range=[82, -2], scaleanchor="x", title="")
fig.show()

## 8. Mini cuadro de mando combinado

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("Puntos acumulados", "Goles a favor y en contra por jornada"))

fig.add_trace(go.Scatter(x=partidos["jornada"], y=partidos["puntos_acum"], mode="lines+markers",
                         name="Puntos"), row=1, col=1)
fig.add_trace(go.Bar(x=partidos["jornada"], y=partidos["goles_favor"], name="A favor"), row=1, col=2)
fig.add_trace(go.Bar(x=partidos["jornada"], y=-partidos["goles_contra"], name="En contra"), row=1, col=2)

fig.update_layout(title_text=f"Dashboard — {EQUIPO}, La Liga 2015/16", barmode="relative")
fig.show()

## 9. Del cuadro de mando al reporting: una tabla por partido

Un informe se diseña para **alguien** y para una **decisión**. Antes de nada preparamos una tabla con una fila por partido y un KPI por columna.

In [ ]:
xg = tiros.groupby(["match_id", "es_sporting"])["shot_xg"].sum().unstack(fill_value=0)
xg.columns = ["xg_contra", "xg_favor"]

pases = (eventos[eventos["type"] == "Pass"]
         .groupby(["match_id", "es_sporting"])["pase_completo"].mean().unstack())
pases.columns = ["pase_rival", "pase_sporting"]

m = (partidos.set_index("match_id").join(xg).join(pases)
     .reset_index().sort_values("jornada"))
m["dif_xg"] = m["xg_favor"] - m["xg_contra"]
m[["jornada", "rival", "local", "resultado", "puntos", "xg_favor", "xg_contra", "dif_xg", "pase_sporting"]].head()

## 10. Semáforo de KPIs

Una cifra sin referencia no dice nada. Definimos un valor, un objetivo y un margen; el estado sale de compararlos.
**Ojo:** el objetivo de 1,4 puntos por partido es un valor de ejemplo, no un objetivo real. Cámbialo y observa qué pasa.

In [ ]:
kpis = pd.DataFrame({
    "KPI": ["Puntos por partido", "Diferencia de xG", "Pases completos (%)"],
    "valor": [m.puntos.mean(), m.dif_xg.mean(), m.pase_sporting.mean() * 100],
    "objetivo": [1.4, 0.0, m.pase_rival.mean() * 100],  # el 1,4 es un objetivo de ejemplo
    "margen": [0.3, 0.3, 3]})

def estado(valor, objetivo, margen):
    if valor >= objetivo: return "verde"
    if valor >= objetivo - margen: return "ámbar"
    return "rojo"

kpis["estado"] = [estado(*f) for f in kpis[["valor", "objetivo", "margen"]].itertuples(index=False)]
kpis.round(2)

## 11. Cuando el informe avisa: verificar, alertar y localizar

**Paso 1 · Comprobar que el dato es fiable.** Un fallo de carga puede parecer un problema deportivo.

In [ ]:
checks = {
    "38 partidos": len(partidos) == 38,
    "sin partidos repetidos": partidos["match_id"].is_unique,
    "sin nulos en puntos": partidos["puntos"].notna().all(),
    "sin eventos duplicados": not eventos.duplicated(["match_id", "event_id"]).any(),
    "todos los partidos tienen eventos": set(partidos["match_id"]) == set(eventos["match_id"]),
}
for nombre, ok in checks.items():
    print("OK   " if ok else "FALLA", nombre)

**Pasos 2 y 3 · Dimensionar y localizar.** Una alerta de racha: 2 puntos o menos en los últimos 5 partidos. La ventana (5) y el umbral (2) son decisiones tuyas: prueba otros valores.

In [ ]:
m["puntos_5"] = m["puntos"].rolling(5).sum()  # puntos de los últimos 5 partidos
m["alerta"] = m["puntos_5"] <= 2

fig = px.line(m, x="jornada", y="puntos_5", markers=True,
              title="Puntos en los últimos 5 partidos")
fig.add_hline(y=2, line_dash="dash", annotation_text="Umbral de alerta")
rojo = m[m["alerta"]]
fig.add_scatter(x=rojo["jornada"], y=rojo["puntos_5"], mode="markers",
                marker=dict(size=14, color="#D1495B"), name="Alerta")
fig.show()

In [ ]:
# ¿Se concentra en algún tipo de partido?
print(m.groupby("local")[["puntos", "xg_favor", "xg_contra", "dif_xg"]].mean().round(2))

# ¿Qué partidos tiran de la media hacia abajo?
cols = ["jornada", "rival", "local", "resultado", "xg_favor", "xg_contra"]
print(m.sort_values("dif_xg").head(5)[cols].round(2))

**Paso 4 · Buscar la causa.** Descarta por este orden: fallo de datos, azar y muestra pequeña, contexto (rival, jugar fuera) y, solo al final, proceso.

**Paso 5 · Actuar y medir.** Toda acción correctiva lleva responsable, plazo y un KPI que diga si ha funcionado.

| Síntoma | Causa a contrastar | Acción | Cómo medir |
|---|---|---|---|
| Pases completos bajos | Salida de balón arriesgada | Trabajar la circulación | % de pases en los 5 partidos siguientes |
| xG en contra alto fuera de casa | Línea defensiva adelantada | Ajustar la altura de la defensa | xG en contra de los próximos partidos fuera |

*Las causas y las acciones son hipótesis de ejemplo para discutir, no conclusiones.*

## 12. Métricas que engañan: los casos del Sporting

Los escenarios de banco y ventas de las slides son **ilustrativos**. Aquí reproducimos la parte que sale de nuestros datos.

In [ ]:
# Caso 1 · El reparto cambia la ratio: pases completos por longitud de pase
pa = eventos[eventos["type"] == "Pass"].copy()
pa["tramo"] = pd.cut(pa["pass_length"], [0, 10, 20, 30, 200], labels=["<10", "10-20", "20-30", ">30"])

print(pa.groupby("es_sporting")["pase_completo"].mean().round(3))
print(pa.groupby("es_sporting")["tramo"].value_counts(normalize=True).unstack().round(3))

mezcla_sporting = pa[pa["es_sporting"]]["tramo"].value_counts(normalize=True)
acierto_rival = pa[~pa["es_sporting"]].groupby("tramo", observed=True)["pase_completo"].mean()
print("Rivales con el reparto del Sporting:", round((acierto_rival * mezcla_sporting).sum(), 3))

In [ ]:
# Caso 2 · Media frente a mediana del xG a favor por partido
xg_pp = xg["xg_favor"]
print("media", round(xg_pp.mean(), 2), "· mediana", round(xg_pp.median(), 2), "· máximo", round(xg_pp.max(), 2))
print("media sin el máximo:", round(xg_pp.drop(xg_pp.idxmax()).mean(), 2))

In [ ]:
# Caso 3 · Acierto con clases raras: ¿qué fracción de los tiros son gol?
goles = (tiros["shot_outcome"] == "Goal")
print(f"{goles.sum()} goles de {len(tiros)} tiros ({goles.mean():.1%}) → acertar siempre «no gol» da {1 - goles.mean():.1%}")

In [ ]:
# Casos 4 y 5 · Actividad frente a valor, y factor de confusión (local o visitante)
tir = tiros[tiros["es_sporting"]].groupby("match_id").size().rename("tiros")
pas = pa[pa["es_sporting"]].groupby("match_id").size().rename("pases")
c = m.set_index("match_id").join(tir).join(pas)

print(c[["puntos", "xg_favor", "tiros", "pases"]].corr()["puntos"].round(2))
for nombre, g in c.groupby("local"):
    print("local" if nombre else "visitante", "· correlación tiros-puntos:", round(g["tiros"].corr(g["puntos"]), 2),
          "· tiros por partido:", round(g["tiros"].mean(), 1), "· puntos por partido:", round(g["puntos"].mean(), 2))

## 📝 Ejercicio propuesto

1. Añade un KPI nuevo: **porcentaje de pases completos** del Sporting en cada partido y su relación con el resultado.
2. ¿En qué partidos el Sporting **perdió pese a crear más peligro** (xG a favor > xG en contra)? ¿Se debe a la mala suerte o a otra causa?
3. Define un **semáforo** para un KPI nuevo: elige la referencia y el umbral, y justifica por qué.
4. Elige una alerta del informe y escribe su **ficha de acción**: causa a contrastar, acción, responsable y KPI para medir si funciona.